# BudgetSwitch W1 Day 6 — GRPO Smoke Test on Kaggle T4

**Goal:** Verify that Unsloth + TRL GRPO pipeline works end-to-end on Kaggle T4 with Qwen2.5-3B-Instruct in 4-bit, completing 5 steps without OOM or exceptions.

**This is NOT real training.** It uses a dummy reward function. The purpose is to confirm the pipeline is set up correctly before we invest 12 GPU-hours in W11.

**Expected runtime:** ~5-10 minutes on Kaggle T4.

**Success criteria:**
- ✅ All cells run without exception
- ✅ 5 GRPO steps complete
- ✅ Peak VRAM ≤ 14 GB
- ✅ Loss is computed (any finite value is acceptable; dummy reward = no real convergence)

**References:**
- Unsloth Qwen2.5 (3B) GRPO notebook: https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide
- Vi-S1K paper (Jan 2026): Vietnamese SLM scaling on Kaggle T4 — proof this resource profile works

## Setup: Install dependencies

Kaggle environments pre-install most things, but Unsloth + latest TRL need explicit install. Pin versions to avoid surprise breakages.

In [ ]:
# Cell 1: Install
# Note: Kaggle's NVIDIA_TF32_OVERRIDE env var may cause issues — disable if needed.
!pip install --quiet unsloth
!pip install --quiet --upgrade trl peft accelerate bitsandbytes
!pip install --quiet vllm  # For fast generation during GRPO rollouts

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Suppress noise

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## Step 1: Load Qwen2.5-3B with Unsloth

Use the pre-quantized 4-bit version to save download time. This is also what we'll use in W9-W11.

In [ ]:
# Cell 2: Load model
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048  # Will be ~4096 for real BudgetSwitch trajectories

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,  # Enables vLLM for fast GRPO rollouts
    gpu_memory_utilization=0.5,  # Leave headroom for training
)

print(f"Model loaded. Peak GPU memory after load: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"Model device: {model.device}")

## Step 2: Add LoRA adapter (mode-routing simulation)

In real BudgetSwitch, this is the mode-routing LoRA (rank 8). Here we use same config to verify the setup.

**Note:** In W9-W11 real training, we'll have TWO LoRAs (tool-use frozen + mode-routing trainable). For smoke test, one LoRA is enough.

In [ ]:
# Cell 3: Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # Mode-routing rank from proposal
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.4f}% of total)")

## Step 3: Build a tiny dummy dataset

10 toy prompts. Not Vietnamese — just verifying pipeline mechanics.

In [ ]:
# Cell 4: Dummy dataset
from datasets import Dataset

DUMMY_PROMPTS = [
    "What is 2+2?",
    "Name the capital of France.",
    "Translate 'hello' to Spanish.",
    "What color is the sky?",
    "How many legs does a spider have?",
    "What is the largest planet?",
    "Who wrote Romeo and Juliet?",
    "What is H2O?",
    "Capital of Japan?",
    "5 times 5?",
]

def format_prompt(example):
    return {
        "prompt": [{"role": "user", "content": example["text"]}],
    }

dataset = Dataset.from_dict({"text": DUMMY_PROMPTS}).map(format_prompt)
print(f"Dataset: {len(dataset)} examples")
print("Sample:", dataset[0])

## Step 4: Dummy reward function

**IMPORTANT:** This is a DUMMY reward for pipeline testing only. The real BudgetSwitch reward (see `training/reward.py` in repo) uses outcome verifiers and follows CRITICAL_RULES.md.

Here: just return random rewards. We just need the pipeline to flow through.

In [ ]:
# Cell 5: Dummy reward
import random

def dummy_reward(prompts, completions, **kwargs):
    """
    Dummy reward for pipeline smoke test.
    
    Real reward function will be in training/reward.py and will:
    1. Parse mode token from completion
    2. Execute step in chosen mode
    3. Run outcome verifier
    4. Compute R = r_outcome - β·cost + γ·r_final
    
    CRITICAL: Real reward MUST NOT reference oracle_mode.
    """
    return [random.uniform(0.0, 1.0) for _ in completions]

# Sanity test: call once
print("Reward test:", dummy_reward(["test"], ["completion"]))

## Step 5: Run GRPO for 5 steps

Small batch, small group size, just to verify mechanics.

In [ ]:
# Cell 6: GRPO training (5 steps)
from trl import GRPOConfig, GRPOTrainer
import time

config = GRPOConfig(
    learning_rate=5e-6,
    num_generations=2,           # Small for smoke test (real: 4)
    max_prompt_length=512,
    max_completion_length=128,   # Small for smoke test
    max_steps=5,                  # JUST 5 STEPS
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=1,
    save_strategy="no",          # Don't save during smoke test
    report_to="none",
    optim="adamw_8bit",
    output_dir="/tmp/grpo_smoke_test",
    beta=0.04,                    # KL penalty (NOT cost weight β)
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=[dummy_reward],
    train_dataset=dataset,
    processing_class=tokenizer,
)

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print(f"\n✅ GRPO smoke test completed in {elapsed:.1f} seconds.")
print(f"   Peak GPU memory during training: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## Success Criteria Verification

Run this final cell and capture screenshot for the W1 retrospective doc.

In [ ]:
# Cell 7: Final verification
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print("=" * 50)
print("W1 Day 6 SMOKE TEST RESULTS")
print("=" * 50)
print(f"GRPO steps completed: 5 ✅")
print(f"Peak VRAM: {peak_vram_gb:.2f} GB  ({'✅ within budget' if peak_vram_gb <= 14 else '⚠️  OVER BUDGET'})")
print(f"Wall time: {elapsed:.1f}s ({elapsed/5:.1f}s per GRPO step)")
print("")
print("Extrapolation to full training (W11):")
print(f"  Real config will have group_size=4 (2x), max_completion=2048 (16x)")
print(f"  Estimated per-step time at full config: {elapsed/5 * 4 * 2:.1f}s")
print(f"  4500 steps × that = {4500 * elapsed/5 * 4 * 2 / 3600:.1f} hours total")
print("")
print("If estimate > 16 GPU-hours, we need to:")
print("  (a) reduce max_completion_length, or")
print("  (b) parallelize across 4 Kaggle accounts, or")
print("  (c) reduce training steps via curriculum.")

## What's Next

If this notebook runs end-to-end:
- ✅ Mark W1 Day 6 milestone done in retrospective
- ✅ Save this notebook to repo `notebooks/W1_grpo_smoke_test.ipynb`
- ✅ Document any version-specific issues in `docs/KAGGLE_NOTES.md`

If it fails:
- Try fallback 1: drop `fast_inference=True` (slower but more stable)
- Try fallback 2: use plain TRL GRPO without Unsloth wrapper
- Try fallback 3: switch to Colab Pro ($10/month, mentioned in Vi-S1K paper)

**Do not proceed to W2 if this smoke test fails.** The pipeline must work before we invest in data construction.